# Main figures


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'output/main_figures').is_dir())
D = ROOT/'output/main_figures'
F = ROOT/'output/figures'
Q = ROOT/'output/figures/checks'
F.mkdir(parents=True, exist_ok=True)
Q.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT/'code'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

OUT=F
from audit_panel_alignment import require_matplotlib_panel_alignment

BLUE, ORANGE, INK, MUTED = '#0072B2', '#D55E00', '#2B3133', '#6E797C'
MINERALS=['Copper','Nickel','PGM','RareEarth','Aluminium','Manganese','Other']
MC=dict(zip(MINERALS,['#567C8D','#5F9E93','#8E719E','#C6A657','#8AA6B8','#B87865','#BEC6C9']))
LABEL={'RareEarth':'Rare earths','Other':'Other minerals','PGM':'PGM'}
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],
 'font.size':7,'axes.labelsize':7,'xtick.labelsize':6.5,'ytick.labelsize':6.5,
 'legend.fontsize':7,'pdf.fonttype':42,'svg.fonttype':'none',
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
 'axes.edgecolor':MUTED,'axes.labelcolor':INK,'text.color':INK,
 'xtick.color':MUTED,'ytick.color':MUTED,'legend.frameon':False})



# Figure 1


In [ ]:
s=pd.read_csv(D/'Figure1a_overview.csv')
fig=plt.figure(figsize=(170/25.4,150/25.4))
gs=fig.add_gridspec(4,2,left=.13,right=.98,bottom=.10,top=.84,
    width_ratios=[1,1],height_ratios=[1,.09,1,.09],hspace=.92,wspace=.22)
a=fig.add_subplot(gs[:,0]); b=fig.add_subplot(gs[0,1]); c=fig.add_subplot(gs[2,1])
sb=fig.add_subplot(gs[1,1]);sc=fig.add_subplot(gs[3,1])
for strip_ax in [sb,sc]:
    pos=strip_ax.get_position()
    strip_height=(188*.026)/150
    strip_ax.set_position([pos.x0,pos.y0+(pos.height-strip_height)/2,pos.width,strip_height])
for ax,letter in zip([a,b,c],'abc'):
    ax.annotate(letter,(0,1),xycoords='axes fraction',xytext=(-17,10),textcoords='offset points',
                fontsize=9,fontweight='bold',annotation_clip=False)
    ax.set_axisbelow(True)
    ax.grid(axis='x' if ax is a else 'y',color='#DDE3E5',linewidth=.5,linestyle=(0,(3,3)))

y=np.arange(len(s),dtype=float); y[-1]+=.5
a.axhspan(y[-1]-.48,y[-1]+.48,color='#F1F4F5',zorder=0)
for key,color,marker,offset in [('worsening',ORANGE,'o',-.13),('improving',BLUE,'D',.13)]:
    a.errorbar(s[key],y+offset,xerr=np.vstack([s[key]-s[key+'_low'],s[key+'_high']-s[key]]),
       fmt=marker,color=color,ecolor=color,markersize=3.8,markeredgewidth=.6,
       elinewidth=.65,capsize=1.5,zorder=3)
a.set_yticks(y,s.label);a.set_ylim(y[-1]+.6,-.65);a.set_xlim(0,85);a.set_xticks([0,20,40,60,80])
a.set_xlabel('Share of trade value undergoing supplier diversification (%)',labelpad=7)

for ax,strip,key,scale,focus,letter,title,xlabel in [
 (b,sb,'delta_origin_hhi',1,.15,'b','Mine-origin concentration','Minimum increase in mine-origin HHI'),
 (c,sc,'delta_top_chokepoint_share',100,40,'c','Maritime exposure','Minimum increase in maritime exposure (percentage points)')]:
    curve=pd.read_csv(D/f'Figure1{letter}_exceedance.csv')
    x=curve.threshold.to_numpy()/scale
    v=curve.value_share_pct.to_numpy()
    ax.plot(x*scale,v,color=INK,linewidth=1.1)
    ax.axvline(.025*scale,color=MUTED,linestyle='--',linewidth=.65)
    ax.set_xlim(0,focus);ax.set_xlabel(xlabel,labelpad=5);ax.set_ylabel('Value share (%)',labelpad=5)
    ax.set_xticks([0,.05,.10,.15] if scale==1 else [0,10,20,30,40])
    ins=ax.inset_axes([.58,.64,.39,.24])
    ins.plot(x*scale,v,color=MUTED,linewidth=.7)
    ins.set_xlim(0,scale);ins.set_ylim(0,4 if scale==1 else 50)
    ins.set_xticks([0,scale]);ins.set_yticks([0,4 if scale==1 else 50])
    ins.tick_params(labelsize=5.5,pad=1,length=2)
    ins.set_title('Full range',fontsize=5.5,pad=3)
    ins.axvspan(0,focus,color='#EDF1F3',zorder=-1)
    composition=pd.read_csv(D/f'Figure1{letter}_minerals.csv')
    fractions=composition.set_index('mineral_group').share_of_layer_affected_value_pct.reindex(MINERALS)
    assert abs(fractions.sum()-100)<1e-8
    left=0
    for mineral,part in fractions.items():
        strip.barh(0,part,left=left,color=MC[mineral],height=.55,edgecolor='none')
        left+=part
    strip.set_xlim(0,100);strip.set_ylim(-.6,.6);strip.set_yticks([]);strip.set_xticks([0,50,100])
    strip.tick_params(axis='x',length=2,pad=2,labelsize=5.5)
    for spine in strip.spines.values():spine.set_visible(False)
    strip.set_title('Mineral composition (%)',fontsize=6,pad=4,loc='left')
b.set_ylim(0,4);b.set_yticks([0,2,4]);c.set_ylim(0,50);c.set_yticks([0,25,50])
handles=[Line2D([],[],marker='o',linestyle='none',color=ORANGE,markersize=4),
         Line2D([],[],marker='D',linestyle='none',color=BLUE,markersize=4)]
outcome_legend=fig.legend(handles,['Origin concentration or maritime exposure increases','Both decrease'],
    loc='upper left',bbox_to_anchor=(.13,.97),ncol=1,labelspacing=.8,handletextpad=.5,
    borderaxespad=0,borderpad=0,fontsize=6)
mineral_legend=fig.legend([Patch(facecolor=MC[m]) for m in MINERALS],[LABEL.get(m,m) for m in MINERALS],
    loc='upper left',bbox_to_anchor=(b.get_position().x0,.97),ncol=4,fontsize=6,
    handlelength=1.0,columnspacing=.75,handletextpad=.4,labelspacing=.8,borderaxespad=0,borderpad=0)
fig.canvas.draw()
# Align the visible bottom of the mineral strip with panel a's x-label bottom.
renderer=fig.canvas.get_renderer()
target_bottom=a.xaxis.label.get_window_extent(renderer).y0
strip_bottom=min(p.get_window_extent(renderer).y0 for p in sc.patches)
delta=(target_bottom-strip_bottom)/fig.bbox.height
for right_ax in [b,c,sb,sc]:
    pos=right_ax.get_position()
    right_ax.set_position([pos.x0,pos.y0+delta,pos.width,pos.height])
# Shorten a from the top, preserving its lower edge and x-label alignment.
pos=a.get_position()
a.set_position([pos.x0,pos.y0,pos.width,b.get_position().y1-pos.y0])
# Both legend rows share the same physical baseline and sit near their panels.
legend_top=b.get_position().y1+.115
outcome_legend.set_bbox_to_anchor((a.get_position().x0,legend_top))
mineral_legend.set_bbox_to_anchor((b.get_position().x0,legend_top))
fig.canvas.draw()
renderer=fig.canvas.get_renderer()
# Keep the aligned top fixed and compress the full right group, including ticks.
target_tick_bottom=a.xaxis.label.get_window_extent(renderer).y0
group_top=b.get_position().y1
for _ in range(8):
    renderer=fig.canvas.get_renderer()
    tick_bottom=min(t.get_window_extent(renderer).y0 for t in sc.get_xticklabels())
    shift=(target_tick_bottom-tick_bottom)/fig.bbox.height
    if abs(shift*fig.bbox.height)*72/fig.dpi < .02:break
    base=sc.get_position().y0
    factor=1-shift/(group_top-base)
    for right_ax in [b,c,sb,sc]:
        pos=right_ax.get_position()
        right_ax.set_position([pos.x0,group_top-(group_top-pos.y0)*factor,pos.width,pos.height*factor])
    fig.canvas.draw()
renderer=fig.canvas.get_renderer()
alignment_error_pt=abs(min(t.get_window_extent(renderer).y0 for t in sc.get_xticklabels())-
    a.xaxis.label.get_window_extent(renderer).y0)*72/fig.dpi
assert alignment_error_pt < .1,alignment_error_pt
(Q/'Figure1_bottom_edge_alignment.json').write_text(json.dumps(dict(
    target='Panel a x-axis label bottom',aligned='Panel c mineral-strip tick-label bottom',
    deviation_pt=alignment_error_pt),indent=2),encoding='utf-8')
# Remove only unused canvas above the legends, preserving physical geometry.
old_height=fig.get_figheight()
new_height=max(outcome_legend.get_window_extent(renderer).y1,
    mineral_legend.get_window_extent(renderer).y1)/fig.dpi+.06
positions={ax:ax.get_position().frozen() for ax in [a,b,c,sb,sc]}
fig.set_size_inches(fig.get_figwidth(),new_height,forward=True)
ratio=old_height/new_height
for ax,pos in positions.items():
    ax.set_position([pos.x0,pos.y0*ratio,pos.width,pos.height*ratio])
outcome_legend.set_bbox_to_anchor((a.get_position().x0,legend_top*ratio))
mineral_legend.set_bbox_to_anchor((b.get_position().x0,legend_top*ratio))
fig.canvas.draw()
require_matplotlib_panel_alignment(fig,json_out=Q/'Figure1.alignment.json',strict=True,
    axes=[a,b,c,sb,sc],panel_ids=['a','b','c','b_strip','c_strip'],
    exemptions=[{'panels':['b','c_strip'],'checks':['row'],
        'reason':'Right group is compressed vertically with its top fixed so its bottom tick labels align to panel a x-label bottom, not plot-area baseline; separately measured to <0.1 pt.'}])
fig.savefig(OUT/'Figure1.pdf')
fig.savefig(OUT/'Figure1.svg')
fig.savefig(OUT/'Figure1.png',dpi=600)
fig.savefig(OUT/'Figure1.tiff',dpi=600,pil_kwargs={'compression':'tiff_lzw'})
plt.close(fig)


In [ ]:
from pathlib import Path
import argparse
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple
from matplotlib.patches import Patch, PathPatch
from matplotlib.path import Path as MPath
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
import shapefile
from shapely.geometry import LineString, Polygon, box

from audit_panel_alignment import require_matplotlib_panel_alignment

INK='#2B3133'; MUTED='#6E797C'; GRID='#DDE3E5'
BLUE='#0072B2'; ORANGE='#D55E00'; TEAL='#008FA5'; GREY='#737D80'
MINERALS=['Copper','Nickel','PGM','RareEarth','Aluminium','Manganese','Other']
MC=dict(zip(MINERALS,['#567C8D','#5F9E93','#8E719E','#C6A657','#8AA6B8','#B87865','#BEC6C9']))
LABEL={'RareEarth':'Rare earths','Other':'Other minerals','PGM':'PGM'}
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'], 'font.size':7,
 'axes.labelsize':7,'axes.titlesize':8,'axes.titleweight':'bold','xtick.labelsize':6,'ytick.labelsize':6,
 'legend.fontsize':6,'pdf.fonttype':42,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False,
 'axes.edgecolor':MUTED,'axes.labelcolor':INK,'text.color':INK,'xtick.color':MUTED,'ytick.color':MUTED,
 'axes.linewidth':.6,'lines.linewidth':1.1,'savefig.facecolor':'white','legend.frameon':False})


def read(name): return pd.read_csv(D/name)
def label(ax, letter, title=''):
    ax.annotate(letter, (0,1), xycoords='axes fraction', xytext=(-18,12), textcoords='offset points', weight='bold', fontsize=9, annotation_clip=False)
    if title: ax.set_title(title, loc='left', pad=12)


def save(fig, number, axes, ids, rows=None, cols=None, exemptions=None):
    fig.canvas.draw()
    options=dict(axes=axes,panel_ids=ids,row_groups=rows or [],column_groups=cols or [])
    if exemptions: options['exemptions']=exemptions
    require_matplotlib_panel_alignment(fig,json_out=Q/f'Figure{number}.alignment.json',strict=True,**options)
    fig.savefig(F/f'Figure{number}.pdf')
    fig.savefig(F/f'Figure{number}.svg')
    fig.savefig(F/f'Figure{number}.png',dpi=600)
    fig.savefig(F/f'Figure{number}.tiff',dpi=600,pil_kwargs={'compression':'tiff_lzw'})
    plt.close(fig)
    print(f'Figure {number} exported',flush=True)


def mineral_legend(fig,y):
    fig.legend([Patch(facecolor=MC[m]) for m in MINERALS],[LABEL.get(m,m) for m in MINERALS],
        loc='upper center',bbox_to_anchor=(.52,y),ncol=7,handlelength=1.15,columnspacing=1.25)


def strip(ax, frame, value_col):
    sub=frame.copy(); sub['group']=sub.metal.where(sub.metal.isin(MINERALS[:-1]),'Other')
    v=sub.groupby('group')[value_col].sum().reindex(MINERALS,fill_value=0)
    v=v/v.sum()*100
    left=0
    for m,x in v.items():
        ax.barh(0,x,left=left,height=.55,color=MC[m],edgecolor='none')
        left+=x
    ax.set(xlim=(0,100),ylim=(-.6,.6),yticks=[],xticks=[0,50,100])
    ax.tick_params(axis='x',length=2,pad=2,labelsize=5.5)
    for s in ax.spines.values():s.set_visible(False)




# Figure 2


In [ ]:
def figure2():
    d=read('Figure2a.csv');cell=read('Figure2b.csv');country=read('Figure2c.csv')
    fig=plt.figure(figsize=(183/25.4,175/25.4))
    a=fig.add_axes([.10,.41,.24,.51]);b=fig.add_axes([.54,.41,.44,.51]);c=fig.add_axes([.10,.11,.88,.16]);cb=fig.add_axes([.65,.325,.30,.012])
    a.plot(d.mismatch_pct,d.year,'o-',ms=3,color=BLUE,label='Supplier–origin mismatch')
    a.plot(d.redistribution_pp,d.year,'s-',ms=3,color=ORANGE,label='Country-share redistribution')
    alternative=read('Figure2a_no_propagation.csv')
    a.plot(alternative.mismatch_pct,alternative.year,'--',color=INK,lw=1,label='Mismatch without propagation')
    a.set(ylim=(2024.5,2011.5),yticks=list(range(2012,2025)),xlim=(0,55),xticks=[0,25,50],xlabel='Share (%) or shift (pp)')
    for start,end in [(2011.5,2014.5),(2019.5,2024.5)]:a.axhspan(start,end,color='#F2F5F6',zorder=0)
    for y in [2014.5,2019.5]:a.axhline(y,color=MUTED,lw=.6,ls='--')
    for year,edition in [(2013,2018),(2017,2021),(2022,2026)]:
        a.text(1.035,year,f'WMD\n{edition}',transform=a.get_yaxis_transform(),fontsize=5.5,color=MUTED,va='center')
    a.legend(loc='upper left',bbox_to_anchor=(-.03,-.095),fontsize=5.5,handlelength=1.7)
    label(a,'a','Annual divergence')
    metals=cell[cell.stage.eq('All stages') & cell.metal.ne('All minerals')].sort_values('mismatch_pct',ascending=False).metal.tolist()+['All minerals']
    stages=['ore','mineral','compound','metal','alloy','magnet','material','All stages']
    matrix=cell.pivot(index='metal',columns='stage',values='mismatch_pct').reindex(index=metals,columns=stages)
    cmap=LinearSegmentedColormap.from_list('mismatch',['#F0F4F5','#B8D7E2','#087BAB']);cmap.set_bad('#C7C9CC')
    im=b.imshow(matrix,aspect='auto',vmin=0,vmax=100,cmap=cmap,interpolation='none')
    b.set(yticks=np.arange(len(metals)),yticklabels=[LABEL.get(m,m) for m in metals],xticks=range(len(stages)),xticklabels=['Ore','Mineral','Compound','Metal','Alloy','Magnet','Material','All stages'])
    plt.setp(b.get_xticklabels(),rotation=40,ha='right',rotation_mode='anchor')
    b.tick_params(length=0,pad=3,labelsize=5.5)
    for x in np.arange(.5,len(stages)-1):b.axvline(x,color='white',lw=.5)
    for y in np.arange(.5,len(metals)-1):b.axhline(y,color='white',lw=.5)
    b.axvline(6.5,color=INK,lw=.8);b.axhline(len(metals)-1.5,color=INK,lw=.8)
    for sp in b.spines.values():sp.set_visible(False)
    label(b,'b','Mismatch by mineral and stage')
    bar=fig.colorbar(im,cax=cb,orientation='horizontal',ticks=[0,25,50,75,100]);bar.set_label('Mismatched attributed value (%)',fontsize=6);bar.outline.set_visible(False)
    fig.legend([Patch(facecolor='#C7C9CC')],['No observed combination'],loc='upper left',bbox_to_anchor=(.39,.345),fontsize=5.5,handlelength=1.2)
    x=np.arange(len(country)); colors=np.where(country.mean_pp.ge(0),BLUE,'#BF5458')
    c.vlines(x,country.min_pp,country.max_pp,color='#B8C3C7',lw=1.4,zorder=1);c.bar(x,country.mean_pp,color=colors,width=.56,zorder=2)
    names={'CHN':'China','ZAF':'South Africa','AUS':'Australia','PER':'Peru','GIN':'Guinea','BRA':'Brazil','NLD':'Netherlands','KOR':'South Korea','GBR':'UK','DEU':'Germany','BEL':'Belgium','JPN':'Japan','RUS':'Russia','IDN':'Indonesia','FIN':'Finland','CAN':'Canada'}
    c.set(xticks=x,xticklabels=[names.get(i,i) for i in country.iso],ylabel='Share difference (pp)')
    plt.setp(c.get_xticklabels(),rotation=36,ha='right',rotation_mode='anchor')
    c.axhline(0,color=INK,lw=.7);c.grid(axis='y',color=GRID,lw=.5);c.set_axisbelow(True)
    label(c,'c','Country roles: attributed origin share minus direct-supplier share')
    fig.text(.10,.025,'Country bars: equal-year means; whiskers: annual range, 2012–2024. Fixed endpoint cohort: 737 units.',fontsize=5.5,color=MUTED)
    save(fig,2,[a,b,c],['a','b','c'],rows=[['a','b']])

figure2()


# Figure 3


In [ ]:
def figure3():
    ie=read('fig3a_temporal_importer_exporter_edges.csv');oe=read('fig3a_temporal_exporter_origin_edges.csv');nodes=read('fig3a_temporal_three_layer_nodes.csv')
    orders=json.loads((D/'Figure3_orders.json').read_text());headers=read('Figure3_changes.csv').set_index('year');mineral=read('Figure3_minerals.csv')
    layers=['inferred mine origin','direct exporter','importer'];titles=['Attributed origins','Direct suppliers','Demand countries']
    fig=plt.figure(figsize=(183/25.4,188/25.4));axes=[];strips=[]
    edge_scale=max(ie.abs_net_usd.max(),oe.abs_net_usd.max());node_scale=nodes.gross_usd.max()
    for i,year in enumerate(range(2021,2025)):
        x=.08+.49*(i%2); y=.53-.43*(i//2)
        ax=fig.add_axes([x,y,.39,.30]);axes.append(ax)
        ax.set(xlim=(-.10,2.10),ylim=(-.12,1.16));ax.axis('off')
        label(ax,chr(97+i),f'{year-1}–{year}')
        fig.text(x,y+.31,f'{int(headers.loc[year,"events"])} events',fontsize=6,color=MUTED,ha='right' if False else 'left')
        positions={}
        for j,layer in enumerate(layers):
            for iso,yy in zip(orders[layer],np.linspace(1,0,len(orders[layer]))):positions[(layer,iso)]=(j,yy)
            ax.text(j,1.15,titles[j],ha='center',fontsize=6)
        for df,src,tar,invert in [(oe,layers[0],layers[1],True),(ie,layers[1],layers[2],True)]:
            for r in df[df.year.eq(year)].itertuples():
                source=r.target if invert else r.source;target=r.source if invert else r.target
                a=positions[(src,source)];b=positions[(tar,target)]
                verts=[(a[0]+.035,a[1]),(a[0]+.42,a[1]),(b[0]-.42,b[1]),(b[0]-.035,b[1])]
                col=ORANGE if r.net_usd>0 else BLUE
                ax.add_patch(PathPatch(MPath(verts,[MPath.MOVETO,MPath.CURVE4,MPath.CURVE4,MPath.CURVE4]),facecolor='none',edgecolor=col,alpha=.65,lw=.3+2.5*np.sqrt(r.abs_net_usd/edge_scale),zorder=1))
        local=nodes[nodes.year.eq(year)]
        for layer in layers:
            for iso in orders[layer]:
                pos=positions[(layer,iso)];row=local[local.layer.eq(layer)&local.node.eq(iso)]
                gross=float(row.gross_usd.sum());net=float(row.net_usd.sum())
                color=GREY if layer=='importer' else (ORANGE if net>0 else BLUE)
                if gross>0:ax.scatter(*pos,s=110*gross/node_scale,c=color,edgecolors='white',linewidths=.5,zorder=3)
                else:ax.plot(*pos,marker='x',ms=2,color='#C9D1D3',mew=.6)
                ax.annotate('OTH' if iso=='Other' else iso,pos,xytext=(0,7),textcoords='offset points',ha='center',fontsize=6,annotation_clip=False)
        st=fig.add_axes([x,y-.055,.39,.026]);strip(st,mineral[mineral.year.eq(year)],'transition_value_usd');strips.append(st)
        fig.text(x,y-.023,'Mineral composition (%)',fontsize=6)
    fig.legend([Line2D([],[],color=ORANGE,lw=2),Line2D([],[],color=BLUE,lw=2),Line2D([],[],marker='o',color=GREY,lw=0),Line2D([],[],marker='x',color='#C9D1D3',lw=0,ms=3)],['Net increase','Net decrease','Demand country','Zero gross change'],loc='upper center',bbox_to_anchor=(.53,.995),ncol=4)
    mineral_legend(fig,.955)
    handles=[Line2D([],[],color=GREY,lw=.3+2.5*np.sqrt(v/edge_scale)) for v in [.25e9,1e9]]
    handles += [Line2D([],[],marker='o',mfc='white',mec=GREY,lw=0,ms=np.sqrt(110*v/node_scale)) for v in [.2e9,1e9]]
    fig.legend(handles,['Link: US$0.25bn','Link: US$1bn','Node: US$0.2bn','Node: US$1bn'],loc='upper center',bbox_to_anchor=(.52,.918),ncol=4)
    fig.text(.08,.874,'Links: absolute net change; nodes: gross change. Orange/blue nodes indicate net increase/decrease.',fontsize=5.5,color=MUTED)
    fig.text(.08,.015,'Conditional origin attribution; event links weighted by end-year trade value. Other countries are bundled as OTH.',fontsize=5.5,color=MUTED)
    ids=['a','b','c','d','a_strip','b_strip','c_strip','d_strip']
    save(fig,3,axes+strips,ids,rows=[['a','b'],['c','d'],['a_strip','b_strip'],['c_strip','d_strip']],cols=[['a','c'],['b','d'],['a_strip','c_strip'],['b_strip','d_strip']])

figure3()


# Figure 4


In [ ]:
def split_wrap(coords):
    segments=[];part=[]
    for lon,lat in coords:
        p=((float(lon)+180)%360-180,float(lat))
        if part and abs(p[0]-part[-1][0])>180:
            if len(part)>1:segments.append(part)
            part=[]
        part.append(p)
    if len(part)>1:segments.append(part)
    return segments

def figure4():
    base=ROOT/'input/map_assets'
    geometry=json.loads((base/'route_geometries.json').read_text())
    world=shapefile.Reader(str(base/'natural_earth/ne_110m_admin_0_countries.shp'),encoding='utf-8')
    polygons=[]; extent=box(-180,-58,180,82)
    for sh in world.shapes():
        points=np.asarray(sh.points);ends=list(sh.parts)+[len(points)]
        for start,stop in zip(ends[:-1],ends[1:]):
            shape=Polygon(points[start:stop])
            if not shape.is_valid:shape=shape.buffer(0)
            clipped=shape.intersection(extent)
            if clipped.is_empty:continue
            parts=list(clipped.geoms) if hasattr(clipped,'geoms') else [clipped]
            polygons.extend(np.asarray(p.exterior.coords) for p in parts if p.geom_type=='Polygon')
    coords={'Malacca':(103.6,1.4),'Hormuz':(56.3,26.6),'Suez':(32.5,30.),'BabAlMandeb':(43.3,12.6),'Dover':(1.4,51.),'Gibraltar':(-5.6,36.),'Panama':(-79.6,9.),'GoodHope':(18.5,-34.4)}
    colors=dict(zip(coords,[BLUE,'#E69F00','#56B4E9',ORANGE,'#CC79A7','#F0E442','#111111','#009E73']))
    names={'BabAlMandeb':'Bab al-Mandeb','GoodHope':'Cape of Good Hope'}
    paths=read('Figure4_displayed_paths.csv');events=read('Figure4_chokepoints.csv');mineral=read('Figure4_mineral_composition.csv');headers=read('Figure4_annual_summary.csv').set_index('year')
    fig=plt.figure(figsize=(183/25.4,155/25.4));axes=[];strips=[];scale=paths.weighted_shift_usd.max()
    vertex_rows=[]
    for i,year in enumerate(range(2021,2025)):
        x=.045+.50*(i%2);y=.53-.40*(i//2)
        ax=fig.add_axes([x,y,.43,.25]);axes.append(ax)
        ax.add_collection(PolyCollection(polygons,facecolors='#F0F3F3',edgecolors='#DDE3E5',linewidths=.18,zorder=0))
        for r in paths[paths.year.eq(year)].sort_values('weighted_shift_usd').itertuples():
            key=f'{r.exporter_iso}|{r.importer_iso}';vertices=geometry[key]
            assert len(vertices)>1,key
            segments=[]
            for part in split_wrap(vertices):
                clipped=LineString(part).intersection(extent)
                if clipped.is_empty:continue
                pieces=list(clipped.geoms) if hasattr(clipped,'geoms') else [clipped]
                segments.extend(np.asarray(p.coords) for p in pieces if p.geom_type=='LineString')
            ax.add_collection(LineCollection(segments,colors='#758188',linewidths=.25+1.1*np.sqrt(r.weighted_shift_usd/scale),alpha=.72,zorder=1))
            for j,(lon,lat) in enumerate(vertices):vertex_rows.append(dict(year=year,exporter=r.exporter_iso,importer=r.importer_iso,chokepoint=r.top_chokepoint,index=j,longitude=lon,latitude=lat))
        e=events[events.year.eq(year)];values=e.groupby('top_chokepoint').transfer_shift_usd.sum();shares=values/values.sum()
        for choke,(lon,lat) in coords.items():
            share=shares.get(choke,0);ax.scatter(lon,lat,s=160*max(.02,share)/.5,color=colors[choke] if share>=.02 else 'white',edgecolors=INK if share>=.02 else colors[choke],linewidths=.5,zorder=3)
        ax.set(xlim=(-180,180),ylim=(-58,82));ax.axis('off')
        label(ax,chr(97+i),f'{year-1}–{year}')
        fig.text(x+.43,y+.280,f'Dominant: {names.get(headers.loc[year,"dominant_chokepoint"],headers.loc[year,"dominant_chokepoint"])}',fontsize=6.5,ha='right')
        st=fig.add_axes([x,y-.053,.43,.025]);strips.append(st);strip(st,mineral[mineral.year.eq(year)],'endpoint_value_usd')
        fig.text(x,y-.020,'Mineral composition (%)',fontsize=6)
        fig.text(x+.43,y-.020,f'{int(headers.loc[year,"event_count"])} events',fontsize=6,color=MUTED,ha='right')
    fig.legend([Line2D([],[],marker='o',color=colors[c],mec=INK,mew=.4,lw=0,ms=4) for c in coords],[names.get(c,c) for c in coords],loc='upper center',bbox_to_anchor=(.52,1.0),ncol=4,columnspacing=2.7)
    mineral_legend(fig,.915)
    handles=[Line2D([],[],color=GREY,lw=.25+1.1*np.sqrt(v/scale)) for v in [100e6,500e6]]
    handles += [Line2D([],[],marker='o',mfc='#BAC3C7',mec=INK,mew=.5,lw=0,ms=np.sqrt(160*v/.5)) for v in [.1,.5]]
    fig.legend(handles,['Exposure increase: US$100m','Exposure increase: US$500m','Chokepoint share: 10%','Chokepoint share: 50%'],loc='upper center',bbox_to_anchor=(.52,.868),ncol=4,columnspacing=1.0,fontsize=5.5)
    fig.text(.045,.020,'Static routes; top 32 paths per window. Mineral bars: event trade-value composition. Hollow circles: share below 2%.',fontsize=5.5,color=MUTED)

    ids=['a','b','c','d','a_strip','b_strip','c_strip','d_strip']
    save(fig,4,axes+strips,ids,rows=[['a','b'],['c','d'],['a_strip','b_strip'],['c_strip','d_strip']],cols=[['a','c'],['b','d'],['a_strip','c_strip'],['b_strip','d_strip']])

figure4()


# Figure 5


In [ ]:
def figure5():
    annual=read('Figure5a.csv');cases=read('Figure5b.csv');draws=read('Figure5a_conditional_draws.csv')
    fig=plt.figure(figsize=(183/25.4,158/25.4));axes=[]
    styles={'direct_partner':(GREY,'s','Direct-only allocation'),'integrated':(TEAL,'o','Three-indicator allocation')}
    for j,(col,title) in enumerate([('risk_transfer_value_share','Risk transfer'),('material_joint_value_share','Three-indicator joint reduction')]):
        ax=fig.add_axes([.12+.49*j,.74,.36,.16]);axes.append(ax)
        for scenario,(color,mark,name) in styles.items():
            for _,sample in draws[draws.scenario.eq(scenario)].groupby(['rho','draw']):
                sample=sample.sort_values('base_year');ax.plot(sample.base_year,100*sample[col],color=color,alpha=.14,lw=.55)
            s=annual[annual.scenario.eq(scenario)].sort_values('base_year')
            ax.plot(s.base_year,100*s[col],color=color,marker=mark,ms=3,mfc='white' if mark=='o' else color,lw=1.2,zorder=5)
        axis_name='Risk transfer\n(value share, %)' if j==0 else 'Three-indicator joint reduction\n(value share, %)'
        ax.set(xlim=(2011.8,2024.2),ylim=(-3,90),xticks=[2012,2018,2024],yticks=[0,40,80],ylabel=axis_name)
        ax.grid(axis='y',color=GRID,lw=.5,linestyle='--')
    label(axes[0],'a')
    handles=[Line2D([],[],color=c,marker=m,ms=3,mfc='white' if m=='o' else c) for c,m,_ in styles.values()]
    handles.append(tuple(Line2D([],[],color=c,alpha=.14,lw=.55) for c,_,_ in styles.values()))
    fig.legend(handles,[n for _,_,n in styles.values()]+['Conditional draws'],
        handler_map={tuple:HandlerTuple(ndivide=None,pad=.25)},
        loc='upper center',bbox_to_anchor=(.54,.985),ncol=3,handlelength=3)
    metrics=['delta_direct_hhi','delta_origin_hhi','delta_top_chokepoint_share'];titles=['Direct-supplier\nconcentration','Attributed mine-origin\nconcentration','Maximum static\nchokepoint exposure']
    metals=['Copper','Aluminium','PGM','Nickel'];density_rows=[]
    for j,(metric,title) in enumerate(zip(metrics,titles)):
        unit_scale=100 if metric=='delta_top_chokepoint_share' else 1
        unit_name='percentage points' if unit_scale==100 else 'index difference'
        ax=fig.add_axes([.17+.275*j,.19,.235,.43]);axes.append(ax)
        for i,metal in enumerate(metals):
            ax.axhline(i,color=GRID,lw=.5,zorder=0)
            for scenario,(color,mark,_) in styles.items():
                s=cases[cases.metal.eq(metal)&cases.scenario.eq(scenario)]
                values=unit_scale*s[metric].to_numpy();weights=s.equal_year_weight.to_numpy()
                assert len(np.unique(values))>1 and np.isfinite(values).all()
                kernel=gaussian_kde(values,weights=weights,bw_method='scott')
                support=np.linspace(max(-unit_scale,values.min()),min(unit_scale,values.max()),401)
                density=kernel(support);height=density/density.max()*.35
                sign=-1 if scenario=='direct_partner' else 1
                ax.fill_between(support,i,i+sign*height,color=color,alpha=.27,lw=0)
                ax.plot(support,i+sign*height,color=color,lw=.8)
                ax.scatter(np.average(values,weights=weights),i+sign*.08,facecolors='white' if scenario=='integrated' else color,marker=mark,s=14,edgecolors=color,linewidths=.8,zorder=4)
                density_rows.extend(dict(metal=metal,scenario=scenario,metric=metric,change_value=x,change_unit=unit_name,density=y) for x,y in zip(support,density))
        ax.axvline(0,color=MUTED,ls='--',lw=.65,zorder=0)
        ax.set(xlim=(-unit_scale,unit_scale),xticks=np.array([-1,-.5,0,.5,1])*unit_scale,ylim=(3.55,-.55),yticks=range(4),yticklabels=metals if j==0 else [])
        ax.set_xticklabels(['−100','−50','0','50','100'] if unit_scale==100 else ['−1','−0.5','0','0.5','1'])
        ax.tick_params(axis='y',length=0,labelsize=7,pad=8)
        axis_label=['Direct-supplier concentration\nChange in concentration index',
                    'Attributed mine-origin concentration\nChange in concentration index',
                    'Maximum static chokepoint\nChange in exposure\n(percentage points)'][j]
        ax.spines['left'].set_visible(False);ax.set_xlabel(axis_label,fontsize=6.5,labelpad=7)
    axes[2].annotate('b',(axes[0].get_position().x0,axes[2].get_position().y1),
        xycoords='figure fraction',xytext=(-18,12),textcoords='offset points',
        weight='bold',fontsize=9,annotation_clip=False)
    fig.canvas.draw()
    panel_labels=[t for ax in axes for t in ax.texts if t.get_text() in ['a','b']]
    assert len(panel_labels)==2
    assert abs(panel_labels[0].get_window_extent().x0-panel_labels[1].get_window_extent().x0)<.1
    pd.DataFrame(density_rows).to_csv(Q/'Figure5b_densities_recomputed.csv',index=False)
    save(fig,5,axes,['a_transfer','a_joint','b_supplier','b_origin','b_route'],rows=[['a_transfer','a_joint'],['b_supplier','b_origin','b_route']])

figure5()
